In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import os
from PIL import Image
from torchvision import transforms

In [ ]:
class ImagesDataset(Dataset):
    def __init__(self, image_folder, transform=None):
        self.image_folder = image_folder
        self.image_paths = [
            os.path.join(image_folder, fname)
            for fname in os.listdir(image_folder)
            if fname.endswith(".JPEG")
        ]
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        img_path = self.image_paths[index]
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, img_path

In [ ]:
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ]
)

In [ ]:
image_folder = "data/test_all"

dataset = ImagesDataset(image_folder, transform)

dataloader = DataLoader(dataset, batch_size=128, shuffle=False)

In [ ]:
from model import ResBlock, ConvNet, LitConvNet

import lightning as L

ckpt_path = "images-classification/xn1ltv41/checkpoints/best-model-epoch=12.ckpt"

base_model = ConvNet(num_classes=50)
model = LitConvNet.load_from_checkpoint(ckpt_path, model=base_model, strict=True)

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.mps.is_available():
    device = "mps"
else:
    device = "cpu"


model.eval()

predictions = []
image_paths = []

with torch.inference_mode():
    for images, paths in dataloader:
        images = images.to(device)

        logits = model(images)
        preds = logits.argmax(dim=1)

        predictions.extend(preds.cpu().numpy())
        image_paths.extend(paths)

In [ ]:
assert len(dataset) == len(predictions)

In [ ]:
import pandas as pd

# os.path.basename(image_paths[0])
results = pd.DataFrame({"Image Path": image_paths, "Class": predictions})
results

In [ ]:
results["Image Path"] = results["Image Path"].apply(os.path.basename)
results

In [ ]:
results.to_csv("pred.csv", index=False, header=False)